This notebook is designed to get parents input while building their account, , estimate the delay, ask milestone questions, come with a support plan and summarize it for parents and the app. 

In [ ]:
# ---------------------------
# Imports & Setups 
# ----------------------------

import os
import re
import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT/"outputs"/"QuestiontoActivity"
OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if OpenAI is None: 
    print("Warning: openai package is not available yet")
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: Openai API key is not visible. LLM backed steps will fail until the key is set")

In [ ]:
# --------------------------------
# Initializations 
# ---------------------------------

DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotional": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",
    "social and emotional": "social_and_emotional",
    "social_emotional": "social_and_emotional",
    "social": "social_and_emotional",
    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",
    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.4,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}


VALID_ANSWERS = set(ANSWER_SCORES.keys())

In [3]:
def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the repo root or notebooks parent folder."
    )

def load_cdc_table(path=None):
    path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [f"{row.category_key}_{row.months}_{i}" for i, row in enumerate(df.itertuples(), start=1)]
    return df, path

cdc_df, cdc_path = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())
print("Loaded CDC file:", cdc_path)
print("CDC ages:", CDC_AGES)
print("Rows:", len(cdc_df))

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

def normalize_answer(answer_text: str) -> str:
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

Loaded CDC file: /Users/sara/Projects/Genex/milestone-cdc-table.xlsx
CDC ages: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows: 159


In [4]:
# ----------------------------------
# State + intake
# ----------------------------------

def new_state() -> Dict[str, Any]:
    return {
        "child": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "support_recommendations": {},
    }

def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    print_banner("INTAKE AGENT")
    name = input("What is your child's name? ").strip()
    age_years = float(input("How old is your child in years? ").strip())
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()

    state["child"] = {
        "name": name,
        "age_years": age_years,
        "chronological_months": int(round(age_years * 12)),
        "diagnosis": diagnosis,
        "concern": concern,
    }

    return state["child"]

In [5]:
# ---------------------------------------
# Delay Estimator 
# ---------------------------------------
def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG[category_key]["display"]

    diagnosis_l = (diagnosis or "").lower()
    concern_l = (concern or "").lower()

    fallback_by_category = {
        "movement_and_physical": 3,
        "language_and_communication": 3,
        "social_and_emotial": 6,
        "cognitive": 6,
    }

    domain_keywords = {
        "movement_and_physical": [
            "motor", "movement", "walk", "run", "jump", "balance", "coordination",
            "fine motor", "gross motor", "grasp", "hand", "writing"
        ],
        "language_and_communication": [
            "speech", "language", "talk", "communication", "words", "sentence",
            "understand", "expressive", "receptive", "verbal"
        ],
        "social_and_emotial": [
            "social", "peer", "friend", "play", "emotion", "emotional",
            "behavior", "anger", "meltdown", "interaction", "turn taking",
            "regulation"
        ],
        "cognitive": [
            "attention", "focus", "concentration", "school", "learning", "routine",
            "executive", "task", "independent", "adaptive", "toilet", "dressing"
        ],
    }

    has_domain_signal = any(
        kw in concern_l for kw in domain_keywords.get(category_key, [])
    )

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    client = OpenAI()

    prompt = f"""
You are a pediatric developmental delay estimator agent for children ages 0 to 5 years.

Your job is to estimate a SINGLE STARTING DELAY in months for one developmental domain only.

This is NOT a diagnosis.
This is NOT a final developmental age.
This is ONLY a rough starting anchor for question selection.

Definition:
delay_months = chronological age in months - estimated functional developmental age in this specific domain

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Instructions:
1. Think only about THIS domain, not overall development.
2. If the diagnosis or concern does NOT meaningfully affect this domain, return 0 to 6 months.
3. If this domain IS affected, estimate the child's functional developmental level in this domain, then convert it into delay_months.
4. Be conservative but realistic.
5. Never exceed the child's chronological age.
6. Return only one integer number of months.
7. Return strict JSON only.

Examples:

Example 1:
Child is 60 months old with Down syndrome.
Domain = Language / Communication.
If you think language skills are functioning around 24 months,
return:
{{"delay_months": 36, "reason": "Language development is likely meaningfully affected in this child."}}

Example 2:
Child is 48 months old with autism.
Domain = Social / Emotional.
If you think social interaction skills are functioning around 24 months,
return:
{{"delay_months": 24, "reason": "Social-emotional development is likely meaningfully affected in this child."}}

Example 3:
Child is 60 months old with cerebral palsy.
Domain = Movement / Physical.
If you think motor skills are functioning around 30 months,
return:
{{"delay_months": 30, "reason": "Movement and physical development are likely meaningfully affected in this child."}}

Example 4:
Child is 30 months old with speech delay.
Domain = Language / Communication.
If you think language skills are functioning around 18 months,
return:
{{"delay_months": 12, "reason": "Language development is likely delayed based on the concern."}}

Example 5:
Child is 36 months old with global developmental delay.
Domain = Cognitive / Adaptive.
If you think adaptive / cognitive functioning is around 18 months,
return:
{{"delay_months": 18, "reason": "Cognitive and adaptive development are likely meaningfully affected."}}

Example 6:
Child is 48 months old with autism.
Domain = Movement / Physical.
If there is no meaningful motor concern,
return:
{{"delay_months": 0, "reason": "There is little reason to think this domain is significantly affected."}}

Example 7:
Child is 60 months old with Down syndrome.
Domain = Social / Emotional.
If you think social skills are only mildly affected,
return:
{{"delay_months": 6, "reason": "This domain may be mildly affected but not severely delayed."}}

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )

        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(parsed.get("delay_months", fallback_by_category.get(category_key, 6)))
        delay_months = max(0, min(delay_months, chronological_months))

        # Guardrail: if the concern does not really point to this domain,
        # do not allow a large delay estimate.
        if not has_domain_signal and delay_months > 6:
            delay_months = 6

        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }

    except Exception as e:
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }
    

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]

In [6]:
# -----------------------------------------
# Milestone Questionnaire 
# ------------------------------------------
def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 4,
    max_bands: int = 3,
    max_questions_total: int = 12,
) -> List[Dict[str, Any]]:
    child = state["child"]
    chrono_months = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    # Available month bands in the chosen window
    available_months = sorted(subset["months"].unique().tolist())

    # Ask nearest bands first
    ordered_months = sorted(
        available_months,
        key=lambda m: (abs(m - approx_dev_months), m)
    )[:max_bands]

    questions = []
    for month in ordered_months:
        month_rows = subset[subset["months"] == month].sort_values("milestone")
        month_rows = month_rows.head(target_questions_per_band)

        for _, row in month_rows.iterrows():
            questions.append({
                "question_id": row["question_id"],
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
            })

    return questions[:max_questions_total]

In [7]:
# ------------------------------------------
# Compute developmental age from the answers
# -------------------------------------------

def compute_dev_age_from_answers(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
) -> int:
    """
    Conservative band-based heuristic.

    A month band is confirmed only if:
    - it has at least `min_yes_confirm` YES answers
    - and YES answers are at least `yes_ratio_confirm` of the answers in that band

    Rules:
    - developmental age = highest confirmed band
    - if no confirmed band, use one lower band than the earliest partial band
    - if only NO answers, use the minimum asked band
    """
    if not answers:
        return 6

    # Group by month band
    band_summary = {}
    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
            }

        band_summary[month]["total"] += 1

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    answered_months = sorted(band_summary.keys())

    confirmed_months = []
    partial_months = []

    for month in answered_months:
        total = band_summary[month]["total"]
        yes_count = band_summary[month]["yes"]
        partial_count = band_summary[month]["partial"]

        yes_ratio = yes_count / total if total > 0 else 0

        if yes_count >= min_yes_confirm and yes_ratio >= yes_ratio_confirm:
            confirmed_months.append(month)
        elif yes_count > 0 or partial_count > 0:
            partial_months.append(month)

    if confirmed_months:
        return int(max(confirmed_months))

    if partial_months:
        anchor = int(min(partial_months))
        lower_candidates = [m for m in answered_months if m < anchor]
        return int(max(lower_candidates)) if lower_candidates else int(anchor)

    return int(min(answered_months))

In [8]:
# -------------------------------------
# Summarize answers by band 
#--------------------------------------

def summarize_answers_by_band(answers: List[Dict[str, Any]]) -> Dict[int, Dict[str, Any]]:
    band_summary = {}

    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
                "items": [],
            }

        band_summary[month]["total"] += 1
        band_summary[month]["items"].append({
            "milestone": a["milestone"],
            "answer": norm,
        })

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    for month in band_summary:
        total = band_summary[month]["total"]
        yes = band_summary[month]["yes"]
        band_summary[month]["yes_ratio"] = round(yes / total, 2) if total else 0.0

    return dict(sorted(band_summary.items()))

In [9]:
# ---------------------------------------
# Question and answer - live 
# ---------------------------------------
def qna_agent_live(
    state: Dict[str, Any],
    category_key: str,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
) -> Dict[str, Any]:
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=4,
        max_bands=3,
        max_questions_total=12,
    )

    asked = []

    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")
        for q in questions_by_month[month]:
            ans = input(q["question_text"] + "\n").strip()
            norm = normalize_answer(ans)
            asked.append({
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "raw_answer": ans,
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
            })

    dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(f"- [{item['months']}m] {item['milestone']} -> {item['norm_answer']}")

    band_summary = summarize_answers_by_band(asked)
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']}"
        )

    print(
        f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
        f"{dev_age} months (chronological age {chrono} months)"
    )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }

In [10]:
# ----------------------------------------
# When there is no special support needed
# ----------------------------------------
def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    dev_age = state["dev_age"].get(category_key, chrono)
    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)

    # Only suppress support when milestone delay is truly minimal
    return (chrono - dev_age) <= 4 and delay_est <= 6

In [11]:
# -------------------------------------
# Select next milestone 
# -------------------------------------
def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    child = state["child"]
    dev_age = state["dev_age"].get(category_key, None)

    if dev_age is None:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key):
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    # Start at the estimated developmental age itself so the plan can include
    # current-level practice plus near next-step skills.
    # We can revise this later if we find the system is still too optimistic
    # or the child needs more reinforcement below the estimated level.
    min_m = max(2, dev_age)
    max_m = min(60, dev_age + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)

    if subset.empty:
        return {
            "status": "success",
            "milestones": [],
        }

    subset = subset.sort_values(["months", "milestone"]).copy()

    milestones = []
    seen = set()

    for _, row in subset.iterrows():
        key = (int(row["months"]), str(row["milestone"]).strip())
        if key in seen:
            continue
        seen.add(key)

        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
        })

    return {
        "status": "success",
        "milestones": milestones[:max_milestones],
    }

In [12]:
# --------------------------------------
# Generate support plan 
# --------------------------------------
def generate_support_plan(state: Dict[str, Any], category_key: str, model: str = "gpt-4o-mini") -> Dict[str, Any]:
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]

    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["support_recommendations"][category_key] = {
            "status": "no_special_support",
            "summary": next_steps["message"],
            "plan": None,
        }
        return state["support_recommendations"][category_key]

    dev_age = state["dev_age"][category_key]

    if next_steps["milestones"]:
        milestone_lines = "\n".join(
            [f"- ({m['months']} months) {m['milestone']}" for m in next_steps["milestones"]]
        )
    else:
        milestone_lines = "- No specific milestone items were found in the selected range. Use the child's profile and category to suggest gentle home practice."

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        plan = {
            "summary": f"Simple fallback home support plan for {category_display}.",
            "day_1": [f"Simple home practice for {category_display.lower()}"],
            "day_2": [f"Repeat with slight variation for {category_display.lower()}"],
            "day_3": [f"Practice during a familiar routine for {category_display.lower()}"],
            "day_4": [f"Use play-based repetition for {category_display.lower()}"],
            "day_5": [f"Review progress and repeat the easiest activity for {category_display.lower()}"],
        }
        state["support_recommendations"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback plan used for {category_display} because OpenAI was unavailable.",
            "plan": plan,
        }
        return state["support_recommendations"][category_key]

    client = OpenAI()

    prompt = f"""
You are a pediatric home-support planning agent helping a parent at home.

This is NOT a diagnosis and NOT a formal treatment plan.
This is a practical, parent-friendly home support plan based on estimated developmental level in one category.

Child:
- Name: {child['name']}
- Chronological age: {child['age_years']} years ({child['chronological_months']} months)
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Estimated developmental age in this category: {dev_age} months

Relevant milestone targets for this category:
{milestone_lines}

Task:
Create a 5-day home support plan with 1 to 2 short, realistic activities per day.

Instructions:
1. Use parent-friendly, practical language.
2. Use activities appropriate for the child's chronological age and estimated developmental level.
3. Focus on practicing current-level skills plus near next-step skills.
4. Do not overclaim.
5. Do not use medical jargon.
6. Do not say "therapy model" or imply guaranteed progress.
7. Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "day_1": ["...", "... optional"],
  "day_2": ["..."],
  "day_3": ["..."],
  "day_4": ["..."],
  "day_5": ["..."]
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "You return strict JSON only and stay non-diagnostic, practical, and parent-friendly."
                },
                {"role": "user", "content": prompt},
            ],
        )

        plan = json.loads(resp.choices[0].message.content)

        state["support_recommendations"][category_key] = {
            "status": "success",
            "summary": plan.get("summary", f"Created a 5-day support plan for {category_display}."),
            "plan": plan,
        }
        return state["support_recommendations"][category_key]

    except Exception as e:
        plan = {
            "summary": f"Fallback home support plan for {category_display}.",
            "day_1": [f"Short home activity for {category_display.lower()}"],
            "day_2": [f"Repeat with slight variation for {category_display.lower()}"],
            "day_3": [f"Practice inside a familiar routine for {category_display.lower()}"],
            "day_4": [f"Use play-based repetition for {category_display.lower()}"],
            "day_5": [f"Review progress for {category_display.lower()}"],
        }
        state["support_recommendations"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback plan used for {category_display} because the OpenAI call failed: {e}",
            "plan": plan,
        }
        return state["support_recommendations"][category_key]

In [13]:
# -------------------------------
# Case summary 
# -------------------------------
def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    child = state["child"]
    rows = []

    for category_key in DOMAIN_CONFIG:
        reco = state["support_recommendations"].get(category_key, {})
        delay_info = state["delay_estimates"].get(category_key, {})
        dev_age = state["dev_age"].get(category_key)

        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if dev_age is None else max(0, chrono_for_gap - dev_age)

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "estimated_dev_age_months": dev_age,
            "milestone_gap_months": milestone_gap,
            "support_status": reco.get("status", "missing"),
            "needs_special_support": False if reco.get("status") == "no_special_support" else True,
        })

    return pd.DataFrame(rows)

In [14]:
# -------------------------
# Display support plans 
# --------------------------
def display_support_plans(state: Dict[str, Any]) -> None:
    child = state["child"]

    print_banner(f"HOME SUPPORT PLANS FOR {child['name']}")

    for category_key in DOMAIN_CONFIG:
        reco = state["support_recommendations"].get(category_key, {})
        category_display = DOMAIN_CONFIG[category_key]["display"]

        print("\n" + "-" * 100)
        print(category_display)
        print("-" * 100)

        if reco.get("status") == "no_special_support":
            print(reco.get("summary", "No special support needed."))
            continue

        plan = reco.get("plan", {})
        if reco.get("summary"):
            print("Summary:", reco["summary"])

        for day in ["day_1", "day_2", "day_3", "day_4", "day_5"]:
            items = plan.get(day, [])
            print(f"\n{day.upper()}:")
            if not items:
                print("- No items")
            else:
                for item in items:
                    print(f"- {item}")

In [15]:
# -------------------------
# Run all - live 
# -------------------------
def run_all_categories_live():
    state = new_state()
    intake_agent_live(state)
    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=2,
            yes_ratio_confirm=0.70,
        )
        generate_support_plan(state, category_key)

    summary_df = summarizer_agent(state)

    # Print plans outside the table
    display_support_plans(state)

    return state, summary_df

In [16]:
# Quick sanity check 
display(cdc_df.head())

,months,category,milestone,category_key,question_id
0,2,social and emotial,calms down when spoken to or picked up,social_and_emotial,social_and_emotial_2_1
1,2,social and emotial,looks at your face,social_and_emotial,social_and_emotial_2_2
2,2,social and emotial,seems happy to see you when you walk up to her,social_and_emotial,social_and_emotial_2_3
3,2,social and emotial,smiles when you talk to or smile at her,social_and_emotial,social_and_emotial_2_4
4,2,language and communication,makes sounds other than crying,language_and_communication,language_and_communication_2_5


In [17]:
# Run fa full live session across all 4 cathegories , age 0-5
state, summary_df = run_all_categories_live()
summary_df


INTAKE AGENT


ValueError: could not convert string to float: 'Ari'

In [ ]:
# ---------------------------------------
# Run 10 test cases 
# ---------------------------------------

cases = [
    {"case_id": "C01", "child_name": "Noah", "age_months": 10, "age_label": "10 months",
     "diagnosis": "Down syndrome",
     "primary_concern": "Not sitting independently yet",
     "secondary_features": "Hypotonia, slow feeding, socially engaged",
     "review_focus": "infant motor + early planning"},

    {"case_id": "C02", "child_name": "Maya", "age_months": 18, "age_label": "18 months",
     "diagnosis": "No formal diagnosis",
     "primary_concern": "No words yet",
     "secondary_features": "Limited pointing, frustration communicating",
     "review_focus": "early language delay"},

    {"case_id": "C03", "child_name": "Leo", "age_months": 26, "age_label": "2y 2m",
     "diagnosis": "Autism spectrum disorder",
     "primary_concern": "Limited eye contact and few words",
     "secondary_features": "Repetitive play, transitions hard, picky eating",
     "review_focus": "social communication + behavior"},

    {"case_id": "C04", "child_name": "Sofia", "age_months": 37, "age_label": "3y 1m",
     "diagnosis": "Prematurity history",
     "primary_concern": "Speech hard to understand",
     "secondary_features": "Fine motor delay, good comprehension",
     "review_focus": "speech intelligibility + fine motor"},

    {"case_id": "C05", "child_name": "Ethan", "age_months": 54, "age_label": "4y 6m",
     "diagnosis": "Cerebral palsy",
     "primary_concern": "Trouble keeping up in preschool",
     "secondary_features": "Self-care delays, frequent falls",
     "review_focus": "motor + adaptive function"},

    {"case_id": "C06", "child_name": "Amina", "age_months": 60, "age_label": "5 years",
     "diagnosis": "Global developmental delay",
     "primary_concern": "Learning and routines are hard",
     "secondary_features": "Short attention span, follows only simple directions",
     "review_focus": "broad developmental support"},

    {"case_id": "C07", "child_name": "Lucas", "age_months": 14, "age_label": "14 months",
     "diagnosis": "Fragile X syndrome",
     "primary_concern": "Not walking, limited babbling",
     "secondary_features": "Sensory sensitivity, shy socially",
     "review_focus": "mixed motor/language/sensory"},

    {"case_id": "C08", "child_name": "Emma", "age_months": 32, "age_label": "2y 8m",
     "diagnosis": "Suspected childhood apraxia of speech",
     "primary_concern": "Very limited speech output",
     "secondary_features": "Uses gestures, gets frustrated, inconsistent sounds",
     "review_focus": "speech disorder specificity"},

    {"case_id": "C09", "child_name": "Zain", "age_months": 48, "age_label": "4 years",
     "diagnosis": "ADHD / attention-regulation concerns",
     "primary_concern": "Very active, impulsive, and struggles to stay with structured activities",
     "secondary_features": "Big emotional reactions, interrupts, difficulty following routines, social interest intact",
     "review_focus": "attention + regulation + functional behavior"},

    {"case_id": "C10", "child_name": "Isla", "age_months": 44, "age_label": "3y 8m",
     "diagnosis": "SYNGAP1-related disorder",
     "primary_concern": "Delayed speech and balance",
     "secondary_features": "Attention issues, clumsy gait, limited pretend play",
     "review_focus": "rare condition usefulness"},
]

df_cases = pd.DataFrame(cases)
print(df_cases)